In [2]:
# MedGraphRAG - Knowledge Graph Builder
# Step 1: Load and explore biomedical data

import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [3]:
from datasets import load_dataset

dataset = load_dataset("qiaojin/PubMedQA", "pqa_labeled", trust_remote_code=True)
df = pd.DataFrame(dataset['train'])
print(df.shape)
print(df.columns.tolist())
df.head(3)

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qiaojin/PubMedQA' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


(1000, 5)
['pubid', 'question', 'context', 'long_answer', 'final_decision']


,pubid,question,context,long_answer,final_decision
0,21645374,Do mitochondria play a role in remodelling lac...,{'contexts': ['Programmed cell death (PCD) is ...,Results depicted mitochondrial dynamics in viv...,yes
1,16418930,Landolt C and snellen e acuity: differences in...,{'contexts': ['Assessment of visual acuity dep...,"Using the charts described, there was only a s...",no
2,9488747,"Syncope during bathing in infants, a pediatric...",{'contexts': ['Apparent life-threatening event...,"""Aquagenic maladies"" could be a pediatric form...",yes


In [4]:
# Explore the data
print("Dataset shape:", df.shape)
print("\nFinal decision distribution:")
print(df['final_decision'].value_counts())

print("\nSample question:")
print(df['question'][0])

print("\nSample context (first abstract):")
print(df['context'][0]['contexts'][0][:300])

Dataset shape: (1000, 5)

Final decision distribution:
final_decision
yes      552
no       338
maybe    110
Name: count, dtype: int64

Sample question:
Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?

Sample context (first abstract):
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel


In [5]:
# Save data locally
import json

# Flatten context for easier processing
records = []
for _, row in df.iterrows():
    records.append({
        'pubid': row['pubid'],
        'question': row['question'],
        'contexts': row['context']['contexts'],
        'long_answer': row['long_answer'],
        'final_decision': row['final_decision']
    })

with open('../data/pubmedqa.json', 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} records to data/pubmedqa.json")

Saved 1000 records to data/pubmedqa.json


In [6]:
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_entities(text):
    doc = nlp(text[:1000])
    entities = []
    for ent in doc.ents:
        if ent.label_ in ["DISEASE", "CHEMICAL", "GENE", "ORG", "PERSON", "GPE"]:
            entities.append({
                "text": ent.text,
                "label": ent.label_
            })
    return entities

# Test on first record
sample_context = records[0]['contexts'][0]
entities = extract_entities(sample_context)
print("Sample context:")
print(sample_context[:300])
print("\nExtracted entities:")
for e in entities:
    print(f"  {e['label']}: {e['text']}")

Sample context:
Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel

Extracted entities:
  GPE: Aponogeton
  ORG: PCD


In [7]:
# Extract entities from all records
print("Extracting entities from all records...")

for i, record in enumerate(records):
    all_entities = []
    for context in record['contexts']:
        entities = extract_entities(context)
        all_entities.extend(entities)
    
    # Deduplicate
    seen = set()
    unique_entities = []
    for e in all_entities:
        key = (e['text'].lower(), e['label'])
        if key not in seen:
            seen.add(key)
            unique_entities.append(e)
    
    record['entities'] = unique_entities
    
    if i % 100 == 0:
        print(f"Processed {i}/1000 records...")

print("Done!")
print(f"Sample entities for record 0: {records[0]['entities'][:5]}")

Extracting entities from all records...
Processed 0/1000 records...
Processed 100/1000 records...
Processed 200/1000 records...
Processed 300/1000 records...
Processed 400/1000 records...
Processed 500/1000 records...
Processed 600/1000 records...
Processed 700/1000 records...
Processed 800/1000 records...
Processed 900/1000 records...
Done!
Sample entities for record 0: [{'text': 'Aponogeton', 'label': 'GPE'}, {'text': 'PCD', 'label': 'ORG'}, {'text': 'MitoTracker Red', 'label': 'ORG'}, {'text': 'TUNEL', 'label': 'ORG'}, {'text': 'PTP', 'label': 'ORG'}]


In [8]:
# Save enriched data with entities
with open('../data/pubmedqa_entities.json', 'w') as f:
    json.dump(records, f, indent=2)

print(f"Saved {len(records)} enriched records")

Saved 1000 enriched records


In [9]:
from neo4j import GraphDatabase
import os
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))
driver.verify_connectivity()
print("Neo4j bağlantısı başarılı!")

Neo4j bağlantısı başarılı!


In [10]:
# Build Knowledge Graph
def create_kg(tx, pubid, question, entity_text, entity_label):
    tx.run("""
        MERGE (d:Document {pubid: $pubid, question: $question})
        MERGE (e:Entity {text: $entity_text, label: $entity_label})
        MERGE (d)-[:HAS_ENTITY]->(e)
    """, pubid=pubid, question=question, 
        entity_text=entity_text, entity_label=entity_label)

print("Loading records into Neo4j...")
with driver.session() as session:
    for i, record in enumerate(records):
        for entity in record['entities']:
            session.execute_write(
                create_kg,
                pubid=str(record['pubid']),
                question=record['question'],
                entity_text=entity['text'],
                entity_label=entity['label']
            )
        if i % 100 == 0:
            print(f"Processed {i}/1000 records...")

print("Knowledge Graph created!")

Loading records into Neo4j...
Processed 0/1000 records...
Processed 100/1000 records...
Processed 200/1000 records...
Processed 300/1000 records...
Processed 400/1000 records...
Processed 500/1000 records...
Processed 600/1000 records...
Processed 700/1000 records...
Processed 800/1000 records...
Processed 900/1000 records...
Knowledge Graph created!
